In [1]:
import os                                                                               # Работа с файловой системой
import sys                                                                              # Системные функции
import math                                                                             # Математические операции
import asyncio                                                                          # Асинхронные функции      
import pickle                                                                           # Сериализация объектов (сохранение моделей)      
import clickhouse_connect                                                               # Драйвер для связи с БД   

import numpy as np                                                                      # Работа с векторами и матрицами           
import polars as pl                                                                     # Работа с таблицами (быстрая)   
import pandas as pd                                                                     # Работа с таблицами (простая)     
import polars.selectors as cs                                                           # Фильтр столбцов polars
import seaborn as sns                                                                   # Простые 2d графики
import matplotlib.pyplot as plt                                                         # Базовая визуализация   
import plotly.graph_objects as go                                                       # Интерактивные 3d графики   

from pathlib import Path                                                                # Работа с путями к файлам
from sklearn import model_selection, decomposition, metrics, linear_model               # Машинное обучение (PCA, logreg, metrics)
from scipy.cluster import hierarchy                                                     # Иерархическая кластеризация
from scipy.spatial import distance, KDTree                                              # Работа с матрицами расстояний
from tqdm import tqdm                                                                   # Отслежевание прогресса
from tqdm.asyncio import tqdm as async_tqdm                                             # Отслежевание прогресса при асинхронных запросах
from importlib.metadata import version                                                  # Вывод информации о библиотеках Python
from collections.abc import Callable                                                    # Подсказки типов
from lib import clickhouse_tools as ch

In [2]:
print(f"Python {sys.version}")
libs = ["clickhouse_connect", "numpy", "pandas", "polars", "seaborn", "matplotlib", "plotly", "scikit-learn", "scipy", "tqdm"]
for lib in libs:
    print(lib, version(lib))


Python 3.14.8 | packaged by conda-forge | (main, Oct  7 2026, 00:23:19) [GCC 15.3.0]
clickhouse_connect 1.10.0
numpy 2.5.3
pandas 3.0.6
polars 2.0.0
seaborn 0.13.2
matplotlib 3.11.2
plotly 7.1.0
scikit-learn 1.9.1
scipy 1.18.1
tqdm 4.70.1


In [3]:
config = ch.Config("../.env")
run_date = "2026-10-08"
step_folder = Path(f"../steps/1/{run_date}")
step_folder.mkdir(parents=True, exist_ok=True)
parameters = {
    "RANDOM_STATE"                    : 9,                             #  Фиксированное состояние генератора случайных чисел (для воспроизводимости)
    "EXCLUDED_CHROMOSOMES"            : ["chrX","chrY","chrM"],        #  Черный список хромосом (фильтр №1)
    "RATE_TOTAL_THRESHOLD"            : 0.8,                           #  Фильтр №2  
    "RATE_CLEAN_DEEP_THRESHOLD"       : 0.5,                           #  Фильтр №3
    "MEDIAN_DEPTH_MIN_THRESHOLD"      : 10,                            #  Фильтр №4 нижняя граница 
    "MEDIAN_DEPTH_MAX_THRESHOLD"      : 100,                           #  Фильтр №4 верхняя граница  
    "STD_BETA_THRESHOLD"              : 20,                            #  Фильтр №5  
    "TOTAL_MIN_AVG_DEPTH"             : 10,                            #  Фильтр качества образцов
    "SSB_QUANTILE_THRESHOLD"          : 0.95,                          #  Фильтр №6 верхняя граница
    "SSW_QUANTILE_THRESHOLD"          : 0.05,                          #  Фильтр №6 нижняя граница
    "MAX_CONCURRENT_QUERIES"          : 4,                             #  Кол-во одновременных запросов к БД
    "IMPUTE_WINDOW_SIZE"              : 500                            #  Размер окна импутации (восстановления) данных
}

## Prepare groups

In [4]:
raw_samplesheet = pl.read_excel("../data/samplesheet.xlsx")
raw_samplesheet

id,btk_short,btk_id,include
str,i64,str,i64
"""K1""",7045200,"""000007045200""",1
"""K2""",7042670,"""000007042670""",1
"""K3""",7103280,"""000007103280""",1
"""K4""",7103250,"""000007103250""",0
"""K5""",7104070,"""000007104070""",0
…,…,…,…
"""N44""",7030340,"""000007030340""",1
"""N43""",7030350,"""000007030350""",1
"""N42""",7030360,"""000007030360""",1


In [5]:
samplesheet = raw_samplesheet.filter(
    pl.col("include") == 1
).with_columns(
    pl.when(pl.col("id").str.starts_with("K"))
    .then(pl.lit("Kabuki"))
    .otherwise(
        pl.when(pl.col("id").str.starts_with("RSTS"))
        .then(pl.lit("RSTS"))
        .otherwise(
            pl.when(pl.col("id").str.starts_with("sot"))
            .then(pl.lit("Sotos"))
            .otherwise(
                pl.lit("Normal")
            )
        )
    )
    .alias("group"),
)
samplesheet

id,btk_short,btk_id,include,group
str,i64,str,i64,str
"""K1""",7045200,"""000007045200""",1,"""Kabuki"""
"""K2""",7042670,"""000007042670""",1,"""Kabuki"""
"""K3""",7103280,"""000007103280""",1,"""Kabuki"""
"""K6""",7044630,"""000007044630""",1,"""Kabuki"""
"""K7""",7027610,"""000007027610""",1,"""Kabuki"""
…,…,…,…,…
"""N44""",7030340,"""000007030340""",1,"""Normal"""
"""N43""",7030350,"""000007030350""",1,"""Normal"""
"""N42""",7030360,"""000007030360""",1,"""Normal"""


In [6]:
samplesheet["group"].value_counts()

group,count
str,u64
"""Kabuki""",18
"""RSTS""",57
"""Normal""",64
"""Sotos""",16


In [7]:
samples_with_db_info = ch.get_samples(config.get_client(), samplesheet["btk_id"].to_list()).with_columns(
    pl.col("btk_id").cast(pl.String).alias("btk_id"),
    percent_valid_sites = pl.col("n_valid_sites") / pl.col("covered_sites")*100    
).join(
    samplesheet, on = "btk_id", how = "inner"
).filter(
    pl.col("mean_depth") >= parameters["TOTAL_MIN_AVG_DEPTH"]
)
samples_with_db_info

ВНИМАНИЕ! Число переданных идентификаторов не совпадает с количеством образцов.


btk_id,medical_record,bam_path,bed_path,birthday,sex,tissue,mean_depth,covered_sites,n_valid_sites,percent_valid_sites,id,btk_short,include,group
str,str,str,str,date,str,str,f32,i32,i32,f64,str,i64,i64,str
"""000007045200""","""3934/2024""","""/media/BTK/2025/05/4-ONT/00000…","""/media/BTK/2025/05/4-ONT/00000…",null,"""Xy""",null,11.143,28732520,23017372,80.109131,"""K1""",7045200,1,"""Kabuki"""
"""000007042670""","""2847/2024""","""/media/BTK/2025/05/5-ONT/00000…","""/media/BTK/2025/05/5-ONT/00000…",null,null,null,11.454,28513646,23218595,81.429765,"""K2""",7042670,1,"""Kabuki"""
"""000007103280""","""20936/2024""","""/media/BTK/2025/06/24-ONT/0000…","""/media/BTK/2025/06/24-ONT/0000…",null,null,null,14.075,28886705,23544074,81.504879,"""K3""",7103280,1,"""Kabuki"""
"""000007044630""","""4940/2024""","""/media/BTK/2025/05/4-ONT/00000…","""/media/BTK/2025/05/4-ONT/00000…",null,"""Xy""",null,10.863,28732268,23176509,80.663695,"""K6""",7044630,1,"""Kabuki"""
"""000007027610""","""24367/2022""","""/media/BTK/2024/06/2-ONT/00000…","""/media/BTK/2024/06/2-ONT/00000…",null,"""Xy""",null,19.652,28880230,23520364,81.441055,"""K7""",7027610,1,"""Kabuki"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""000007030290""","""32004/2023""","""/media/BTK/2024/06/21-ONT/0000…","""/media/BTK/2024/06/21-ONT/0000…",null,"""XX""",null,12.41,28859724,23083381,79.98476,"""N45""",7030290,1,"""Normal"""
"""000007030340""","""24683/2020""","""/media/BTK/2024/06/21-ONT/0000…","""/media/BTK/2024/06/21-ONT/0000…",null,"""Xy""",null,14.302,28857685,23812315,82.516373,"""N44""",7030340,1,"""Normal"""
"""000007030350""","""24683/2020""","""/media/BTK/2024/06/21-ONT/0000…","""/media/BTK/2024/06/21-ONT/0000…",null,"""XX""",null,15.748,28724175,23491388,81.782638,"""N43""",7030350,1,"""Normal"""


In [8]:
samples_with_db_info["group"].value_counts()

group,count
str,u64
"""Normal""",59
"""Sotos""",15
"""Kabuki""",17
"""RSTS""",18


In [9]:
to_split = samples_with_db_info.to_pandas()
sample_train, sample_val = model_selection.train_test_split(to_split, test_size = 0.3, stratify = to_split["group"], random_state=parameters["RANDOM_STATE"])
sample_train["group"].value_counts()

group
Normal    41
RSTS      13
Kabuki    12
Sotos     10
Name: count, dtype: int64

In [10]:
sample_val["group"].value_counts()


group
Normal    18
Sotos      5
RSTS       5
Kabuki     5
Name: count, dtype: int64

In [11]:
sample_train = sample_train.sort_values("btk_id")
sample_val = sample_val.sort_values("btk_id")
samples_with_db_info_sorted = samples_with_db_info.to_pandas().sort_values("btk_id")
sample_train.to_csv(step_folder / "train_samples.csv", index = False)
sample_val.to_csv(step_folder / "val_samples.csv", index = False)
samples_with_db_info_sorted.to_csv(step_folder / "total_info.csv", index = False)

## Filtering

In [12]:
cpg = pl.read_csv("../data/CpG.bed",separator="\t",has_header=True).\
    select(
        ["seqnames","start"]
    ).\
    filter(
        ~pl.col("seqnames").is_in(parameters['EXCLUDED_CHROMOSOMES'])
    )
cpg.columns = ["chr","start"]
print(cpg.shape)
cpg.head()

(27852739, 2)


chr,start
str,i64
"""chr1""",10468
"""chr1""",10470
"""chr1""",10483
"""chr1""",10488
"""chr1""",10492


In [13]:
semaphore = asyncio.Semaphore(parameters['MAX_CONCURRENT_QUERIES'])
batches = ch.get_batches(cpg, batch_size = 100000)    
    
requests = [ch.task(ch.get_descriptive_statistics, semaphore, batch, i, sample_train["btk_id"].to_list(), parameters, config) for i, batch in batches]
results = await async_tqdm.gather(*requests)
list_stats = [df for _, df in results if df is not None]
descriptive = pl.concat(list_stats)
descriptive.write_csv(step_folder / "locus_descriptive_statistics.csv")
descriptive

100%|██████████| 290/290 [16:16<00:00,  3.37s/it]


chr,pos,not_null,n_clean_deep,beta_mean,beta_std,median_depth
str,u32,u64,u64,f64,f64,f64
"""chr1""",916945,76,39,26.475132,20.534152,12.0
"""chr1""",868039,76,49,57.041316,21.051241,13.0
"""chr1""",3323244,76,40,19.587368,20.345529,11.5
"""chr1""",1759017,76,47,38.093421,25.805114,14.0
"""chr1""",3120391,76,60,56.678553,23.887965,13.5
…,…,…,…,…,…,…
"""chr22""",50547479,76,48,49.9525,24.948064,14.0
"""chr22""",50387685,76,66,58.868684,20.095565,18.0
"""chr22""",49634562,76,46,54.336842,21.294253,13.0


In [14]:
semaphore = asyncio.Semaphore(parameters['MAX_CONCURRENT_QUERIES'])
batches = ch.get_batches(descriptive, batch_size = 10000)    
    
requests = [ch.task(ch.get_methylation_data, semaphore, batch, i, sample_train["btk_id"].to_list(), config) for i, batch in batches]
results = await async_tqdm.gather(*requests)
list_res = [df for _, df in results if df is not None]
for i in range(1,len(list_res)):
    list_res[i] = list_res[i].drop("btk_id")# Удалим столбец с идентификаторами из всех порций кроме первой
data_train_raw = pl.concat(list_res,how="horizontal")
data_train_raw.write_csv(step_folder / "data_train_raw.csv")
data_train_raw

100%|██████████| 33/33 [06:29<00:00, 11.81s/it]


btk_id,chr1_21607,chr1_21615,chr1_21685,chr1_22245,chr1_22348,chr1_51441,chr1_51934,chr1_52364,chr1_5857943,chr1_5857953,chr1_5890222,chr1_5890227,chr1_5933855,chr1_53833,chr1_61019,chr1_62419,chr1_75343,chr1_90536,chr1_437144,chr1_478749,chr1_596267,chr1_596371,chr1_605136,chr1_605224,chr1_605231,chr1_605237,chr1_5975611,chr1_5975626,chr1_5975666,chr1_5975776,chr1_5975779,chr1_5975784,chr1_5975943,chr1_5975961,chr1_5976032,chr1_5980560,…,chr22_47049626,chr22_47049909,chr22_43835140,chr22_43835149,chr22_43835168,chr22_43835177,chr22_43852670,chr22_43862145,chr22_43862482,chr22_43862526,chr22_43862536,chr22_43862571,chr22_43876569,chr22_43880345,chr22_43889931,chr22_47066072,chr22_47071529,chr22_47091459,chr22_47091593,chr22_47091601,chr22_47124027,chr22_47129992,chr22_47148891,chr22_47148942,chr22_47148967,chr22_47148983,chr22_47149028,chr22_47149097,chr22_47149100,chr22_47149202,chr22_47149212,chr22_47149256,chr22_47149264,chr22_47149267,chr22_47149279,chr22_47149283,chr22_47149298
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""000007028770""",0.13,0.13,0.31,0.31,0.5,null,null,null,0.4,0.25,0.5,0.29,0.25,null,null,null,null,null,0.29,0.33,0.33,0.75,0.5,0.43,0.43,0.38,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.08,0.0,0.09,…,0.92,0.08,0.56,0.56,0.62,0.6,0.11,0.0,0.15,0.0,0.0,0.08,1.0,0.25,0.43,0.53,0.08,0.33,0.3,0.5,0.21,0.67,0.42,0.31,0.14,0.08,0.15,0.31,0.18,0.14,0.14,0.2,0.11,0.25,0.46,0.42,0.15
"""000007028880""",0.27,0.46,0.46,0.31,0.62,1.0,1.0,1.0,0.62,0.67,0.5,0.67,0.28,1.0,null,null,null,1.0,0.33,0.47,0.8,0.75,0.62,0.0,0.5,0.5,0.5,0.43,0.43,0.38,0.29,0.38,0.43,0.33,0.33,0.25,…,0.73,0.3,0.11,0.11,0.11,0.11,0.2,0.1,0.11,0.1,0.1,0.1,null,0.53,0.12,0.0,0.0,0.6,0.89,0.79,0.26,0.56,0.42,0.18,0.08,0.08,0.09,0.33,0.42,0.23,0.23,0.17,0.38,0.36,0.6,0.5,0.36
"""000007029030""",0.45,0.35,0.4,0.47,0.88,0.89,0.67,0.26,0.25,0.56,0.69,0.75,0.33,0.82,0.71,0.5,0.41,0.68,0.5,0.27,0.8,0.78,0.0,0.0,0.0,0.0,0.5,0.32,0.39,0.33,0.41,0.39,0.61,0.57,0.4,0.28,…,0.8,0.33,0.38,0.38,0.38,0.38,0.67,0.2,0.0,0.06,0.12,0.17,0.71,0.08,0.88,0.62,0.17,0.27,0.56,0.82,0.29,0.3,0.75,0.55,0.6,0.43,0.65,0.58,0.61,0.64,0.52,0.62,0.68,0.63,0.72,0.68,0.71
"""000007029040""",0.47,0.44,0.55,0.5,0.71,0.86,0.83,0.62,0.5,0.47,0.29,0.27,0.16,1.0,0.64,0.58,0.0,0.5,0.62,0.33,1.0,1.0,0.33,0.33,0.33,0.33,0.4,0.29,0.31,0.29,0.29,0.31,0.57,0.42,0.08,0.43,…,0.6,0.27,0.53,0.54,0.6,0.56,0.33,0.0,0.0,0.0,0.09,0.0,0.6,0.6,0.75,0.67,0.47,0.64,0.58,0.58,0.5,0.38,0.88,0.83,0.76,0.82,0.63,0.71,0.71,0.81,0.72,0.8,0.88,0.93,0.75,0.93,0.87
"""000007029220""",0.43,0.46,0.38,0.31,0.8,0.73,0.91,0.25,0.67,0.57,0.33,0.4,0.0,0.67,0.25,0.4,0.5,0.8,0.67,0.33,0.64,0.67,0.45,0.55,0.45,0.5,0.33,0.3,0.33,0.33,0.25,0.3,0.33,0.33,0.3,0.17,…,0.64,0.4,0.5,0.33,0.5,0.5,0.29,0.09,0.0,0.07,0.07,0.0,1.0,0.2,1.0,0.55,0.13,0.78,0.67,0.8,0.36,0.5,0.67,0.12,0.4,0.4,0.5,0.5,0.5,0.5,0.6,0.7,0.9,0.75,0.88,0.89,0.7
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""000027007830""",0.26,0.24,0.37,0.33,0.53,0.75,0.5,0.4,0.33,0.25,0.42,0.28,0.08,0.5,1.0,0.62,0.22,0.6,0.55,0.58,1.0,1.0,0.18,0.09,0.09,0.0,0.53,0.33,0.24,0.36,0.3,0.29,0.25,0.25,0.27,0.53,…,0.48,0.36,0.92,0.85,0.8,0.83,0.55,0.17,0.0,0.0,0.0,0.0,0.64,0.25,0.3,0.65,0.12,0.23,0.52,0.5,0.18,0.53,0.88,0.55,0.44,0.47,0.57,0.45,0.26,0.5,0.56,0.77,0.64,0.6,0.53,0.4,0.6
"""000027007840""",0.35,0.42,0.24,0.25,0.4,0.76,0.81,0.13,0.57,0.71,0.62,0.41,0.13,0.73,0.73,0.5,0.55,0.64,0.6,0.62,0.88,0.57,0.36,0.5,0.45,0.33,0.33,0.5,0.5,0.6,0.5,0.5,0.73,0.83,0.38,0.25,…,0.67,0.43,0.82,0.7,0.85,0.79,0.25,null,0.0,0.0,0.0,0.0,0.0,0.5,0.64,0.57,0.86,0.0,0.0,0.0,0.2,0.29,0.12,0.1,0.0,0.0,0.29,0.2,0.0,0.0,0.0,0.25,0.12,0.0,0.12,0.33,0.1
"""0000

In [15]:
data_train_full = data_train_raw.select([col for col in data_train_raw.columns if data_train_raw[col].null_count() == 0])
data_train_full.write_csv(step_folder / "data_train_raw.csv")
data_train_full

btk_id,chr1_21607,chr1_21615,chr1_21685,chr1_22245,chr1_22348,chr1_5857943,chr1_5857953,chr1_5890222,chr1_5890227,chr1_5933855,chr1_437144,chr1_478749,chr1_5975626,chr1_5975666,chr1_5975776,chr1_5975779,chr1_5975784,chr1_5975943,chr1_5975961,chr1_5976032,chr1_5980560,chr1_21688310,chr1_21691582,chr1_21691612,chr1_21709545,chr1_21756563,chr1_21756616,chr1_12689156,chr1_12689160,chr1_12691090,chr1_12693445,chr1_12693638,chr1_12721301,chr1_5994430,chr1_6007443,chr1_6007468,…,chr22_46961626,chr22_46961786,chr22_46961804,chr22_46961819,chr22_46961828,chr22_46962000,chr22_46962067,chr22_47049626,chr22_47049909,chr22_43835140,chr22_43835149,chr22_43835168,chr22_43835177,chr22_43852670,chr22_43880345,chr22_47066072,chr22_47071529,chr22_47091459,chr22_47091593,chr22_47091601,chr22_47124027,chr22_47129992,chr22_47148891,chr22_47148942,chr22_47148967,chr22_47148983,chr22_47149028,chr22_47149097,chr22_47149100,chr22_47149202,chr22_47149212,chr22_47149256,chr22_47149264,chr22_47149267,chr22_47149279,chr22_47149283,chr22_47149298
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""000007028770""",0.13,0.13,0.31,0.31,0.5,0.4,0.25,0.5,0.29,0.25,0.29,0.33,0.0,0.0,0.0,0.0,0.0,0.0,0.08,0.0,0.09,0.55,0.14,0.0,0.44,0.35,0.21,0.31,0.27,0.33,0.06,0.46,0.5,0.44,0.73,0.62,…,0.12,0.45,0.21,0.54,0.24,0.0,0.29,0.92,0.08,0.56,0.56,0.62,0.6,0.11,0.25,0.53,0.08,0.33,0.3,0.5,0.21,0.67,0.42,0.31,0.14,0.08,0.15,0.31,0.18,0.14,0.14,0.2,0.11,0.25,0.46,0.42,0.15
"""000007028880""",0.27,0.46,0.46,0.31,0.62,0.62,0.67,0.5,0.67,0.28,0.33,0.47,0.43,0.43,0.38,0.29,0.38,0.43,0.33,0.33,0.25,0.56,0.3,0.17,0.38,0.5,0.47,0.07,0.07,0.0,0.0,0.0,0.5,0.16,0.73,0.33,…,0.47,0.67,0.57,0.69,0.6,0.5,0.33,0.73,0.3,0.11,0.11,0.11,0.11,0.2,0.53,0.0,0.0,0.6,0.89,0.79,0.26,0.56,0.42,0.18,0.08,0.08,0.09,0.33,0.42,0.23,0.23,0.17,0.38,0.36,0.6,0.5,0.36
"""000007029030""",0.45,0.35,0.4,0.47,0.88,0.25,0.56,0.69,0.75,0.33,0.5,0.27,0.32,0.39,0.33,0.41,0.39,0.61,0.57,0.4,0.28,0.53,0.2,0.18,0.29,0.47,0.4,0.44,0.4,0.55,0.33,0.58,0.6,0.44,0.67,0.71,…,0.2,0.31,0.14,0.2,0.13,0.2,0.21,0.8,0.33,0.38,0.38,0.38,0.38,0.67,0.08,0.62,0.17,0.27,0.56,0.82,0.29,0.3,0.75,0.55,0.6,0.43,0.65,0.58,0.61,0.64,0.52,0.62,0.68,0.63,0.72,0.68,0.71
"""000007029040""",0.47,0.44,0.55,0.5,0.71,0.5,0.47,0.29,0.27,0.16,0.62,0.33,0.29,0.31,0.29,0.29,0.31,0.57,0.42,0.08,0.43,0.38,0.5,0.23,0.28,0.6,0.5,0.57,0.54,0.3,0.36,0.89,0.73,0.25,0.76,0.79,…,0.11,0.22,0.17,0.17,0.17,0.12,0.2,0.6,0.27,0.53,0.54,0.6,0.56,0.33,0.6,0.67,0.47,0.64,0.58,0.58,0.5,0.38,0.88,0.83,0.76,0.82,0.63,0.71,0.71,0.81,0.72,0.8,0.88,0.93,0.75,0.93,0.87
"""000007029220""",0.43,0.46,0.38,0.31,0.8,0.67,0.57,0.33,0.4,0.0,0.67,0.33,0.3,0.33,0.33,0.25,0.3,0.33,0.33,0.3,0.17,0.47,0.5,0.22,0.45,0.67,0.33,0.2,0.2,0.0,0.0,0.6,0.12,0.22,0.38,0.44,…,0.38,0.36,0.14,0.36,0.3,0.4,0.22,0.64,0.4,0.5,0.33,0.5,0.5,0.29,0.2,0.55,0.13,0.78,0.67,0.8,0.36,0.5,0.67,0.12,0.4,0.4,0.5,0.5,0.5,0.5,0.6,0.7,0.9,0.75,0.88,0.89,0.7
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""000027007830""",0.26,0.24,0.37,0.33,0.53,0.33,0.25,0.42,0.28,0.08,0.55,0.58,0.33,0.24,0.36,0.3,0.29,0.25,0.25,0.27,0.53,0.53,0.18,0.04,0.24,0.45,0.28,0.26,0.35,0.29,0.32,0.38,0.38,0.75,0.77,0.8,…,0.28,0.38,0.24,0.29,0.2,0.23,0.38,0.48,0.36,0.92,0.85,0.8,0.83,0.55,0.25,0.65,0.12,0.23,0.52,0.5,0.18,0.53,0.88,0.55,0.44,0.47,0.57,0.45,0.26,0.5,0.56,0.77,0.64,0.6,0.53,0.4,0.6
"""000027007840""",0.35,0.42,0.24,0.25,0.4,0.57,0.71,0.62,0.41,0.13,0.6,0.62,0.5,0.5,0.6,0.5,0.5,0.73,0.83,0.38,0.25,0.4,0.18,0.08,0.56,0.18,0.2,0.15,0.14,0.29,0.11,0.4,0.44,0.76,0.41,0.6,…,0.3,0.57,0.44,0.44,0.5,0.5,0.5,0.67,0.43,0.82,0.7,0.85,0.79,0.25,0.5,0.57,0.86,0.0,0.0,0.0,0.2,0.29,0.12,0.1,0.0

In [16]:
btk_id = data_train_full["btk_id"].to_list()
locuses = data_train_full.columns[1:]
data_train_full_T = data_train_full.drop("btk_id").transpose()
data_train_full_T.columns = btk_id
data_train_full_T = pl.concat([data_train_full_T, pl.DataFrame(locuses,schema={"locus":pl.String})],how = "horizontal")
normal_id = sample_train.loc[sample_train["group"] == "Normal","btk_id"].to_list()
kabuki_id = sample_train.loc[sample_train["group"] == "Kabuki","btk_id"].to_list()
sotos_id = sample_train.loc[sample_train["group"] == "Sotos","btk_id"].to_list()
rsts_id = sample_train.loc[sample_train["group"] == "RSTS","btk_id"].to_list()
data_train_full_T_RSS = data_train_full_T.with_columns(
    pl.mean_horizontal(pl.all().exclude(["locus"])).alias("mean_total"),
    pl.mean_horizontal(normal_id).alias("mean_normal"),
    pl.mean_horizontal(kabuki_id).alias("mean_kabuki"),
    pl.mean_horizontal(sotos_id).alias("mean_sotos"),
    pl.mean_horizontal(rsts_id).alias("mean_rsts")    
).with_columns(
    SSW = pl.sum_horizontal([(pl.col(c) - pl.col("mean_normal")).pow(2) for c in normal_id]) + pl.sum_horizontal([(pl.col(c) - pl.col("mean_kabuki")).pow(2) for c in kabuki_id]) + pl.sum_horizontal([(pl.col(c) - pl.col("mean_sotos")).pow(2) for c in sotos_id]) + pl.sum_horizontal([(pl.col(c) - pl.col("mean_rsts")).pow(2) for c in rsts_id]),
    SSB = (len(normal_id) * (pl.col("mean_normal") - pl.col("mean_total")).pow(2)) + (len(kabuki_id) * (pl.col("mean_kabuki") - pl.col("mean_total")).pow(2)) + (len(sotos_id) * (pl.col("mean_sotos") - pl.col("mean_total")).pow(2)) + (len(rsts_id) * (pl.col("mean_rsts") - pl.col("mean_total")).pow(2))
)
low_SSW_high_SSB = data_train_full_T_RSS.filter(
    (pl.col("SSW") <= pl.col("SSW").quantile(parameters['SSW_QUANTILE_THRESHOLD'])) & (pl.col("SSB") >= pl.col("SSB").quantile(parameters['SSB_QUANTILE_THRESHOLD']))
)
low_SSW_high_SSB

000007028770,000007028880,000007029030,000007029040,000007029220,000007029250,000007029310,000007029350,000007029580,000007029610,000007029640,000007029690,000007029750,000007029790,000007029820,000007029850,000007029860,000007029870,000007029910,000007029920,000007029930,000007029940,000007029950,000007030020,000007030050,000007030080,000007030100,000007030110,000007030120,000007030140,000007030150,000007030180,000007030190,000007030200,000007030210,000007030230,000007030260,…,000007171740,000007172340,000007175430,000007175440,000007175450,000007175470,000027000110,000027000160,000027000190,000027005730,000027005740,000027005770,000027005800,000027005810,000027005820,000027005880,000027007520,000027007680,000027007740,000027007750,000027007770,000027007780,000027007790,000027007820,000027007830,000027007840,000027007900,000027007990,000027008020,locus,mean_total,mean_normal,mean_kabuki,mean_sotos,mean_rsts,SSW,SSB
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64
0.9,0.93,1.0,1.0,0.85,0.94,1.0,0.95,1.0,1.0,0.79,0.86,1.0,1.0,0.81,0.97,0.73,1.0,0.93,0.92,1.0,1.0,0.89,0.94,0.81,1.0,1.0,1.0,0.94,0.85,1.0,0.95,0.83,1.0,1.0,0.67,1.0,…,0.79,0.76,1.0,1.0,0.91,0.93,0.14,0.29,0.12,0.18,0.3,0.38,0.13,0.27,0.0,0.13,1.0,0.94,1.0,0.83,0.94,1.0,0.95,0.88,0.96,1.0,1.0,0.95,0.96,"""chr1_21784097""",0.838816,0.93439,0.924167,0.194,0.954615,0.506864,4.794129
0.84,0.93,0.9,1.0,0.82,0.94,0.95,0.94,1.0,0.94,0.8,0.84,1.0,1.0,0.83,0.81,0.75,0.82,0.92,0.86,1.0,0.94,0.93,0.93,0.71,0.95,1.0,0.92,1.0,0.78,1.0,1.0,0.76,1.0,0.85,0.67,1.0,…,0.7,0.77,1.0,0.95,0.94,0.92,0.17,0.29,0.17,0.15,0.18,0.42,0.08,0.21,0.0,0.17,1.0,0.93,1.0,0.91,0.94,1.0,0.94,0.85,0.95,1.0,0.82,0.89,0.95,"""chr1_21784104""",0.810658,0.904634,0.875,0.184,0.936923,0.584636,4.546031
0.6,0.47,0.78,0.55,0.45,0.63,0.71,0.77,0.71,0.75,0.85,0.62,0.67,0.8,0.77,0.56,0.75,0.83,0.71,0.82,1.0,0.62,0.68,0.71,0.56,0.79,0.55,0.75,0.65,0.62,0.84,0.91,0.56,0.67,0.71,0.64,0.69,…,0.55,0.72,0.87,0.68,0.7,0.75,0.0,0.11,0.29,0.0,0.43,0.3,0.0,0.0,0.0,0.0,0.83,0.88,1.0,0.89,0.88,0.94,0.94,0.88,0.92,1.0,0.73,0.75,0.95,"""chr1_21784141""",0.660921,0.699268,0.736667,0.113,0.891538,1.051924,3.822712
0.82,0.75,0.89,0.73,0.55,0.89,0.96,0.9,0.86,0.95,0.93,0.71,1.0,0.84,0.83,0.84,0.83,0.95,0.87,0.93,0.95,0.81,0.85,0.86,0.75,0.96,0.9,1.0,0.94,0.86,0.94,0.92,0.79,0.88,0.96,0.83,0.93,…,0.76,0.77,1.0,1.0,0.82,0.97,0.0,0.33,0.25,0.25,0.57,0.5,0.17,0.44,0.23,0.33,1.0,0.94,1.0,0.92,0.94,1.0,0.94,1.0,0.92,1.0,0.82,0.94,0.96,"""chr1_21784150""",0.812632,0.866585,0.898333,0.307,0.952308,0.700929,3.017744
0.86,0.88,0.81,0.73,0.75,0.94,0.91,0.87,0.75,0.76,0.92,0.79,0.83,0.8,0.74,0.72,0.8,0.83,0.73,0.79,0.84,0.83,0.84,0.84,0.76,0.92,0.78,0.88,0.88,0.67,0.94,0.96,0.67,0.8,0.83,1.0,1.0,…,0.81,0.73,0.88,0.95,0.91,0.88,0.12,0.21,0.12,0.07,0.25,0.29,0.04,0.14,0.0,0.18,0.83,0.61,1.0,0.85,0.81,0.82,0.82,0.71,0.88,0.92,0.82,0.81,0.89,"""chr1_21784204""",0.748947,0.836585,0.869167,0.142,0.828462,0.547143,4.254373
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.5,0.5,0.75,0.69,0.8,0.6,0.76,0.36,0.72,0.62,0.69,0.83,0.67,0.64,0.5,0.29,0.62,0.71,0.62,0.85,0.71,0.69,0.8,0.5,0.81,0.82,0.64,0.86,0.61,0.46,0.65,0.7,0.85,0.25,1.0,0.73,0.82,…,0.63,0.88,0.71,0.87,0.81,0.66,0.0,0.25,0.0,0.21,0.09,0.1,0.12,0.0,0.0,0.22,0.71,0.67,0.82,0.88,0.64,0.45,0.4,0.72,0.84,0.67,0.75,0.81,0.9,"""chr22_44273391""",0.607237,0.656341,0.749167,0.099,0.712308,1.470564,3.067156
0.33,0.89,1.0,0.85,0.92,0.91,0.86,0.92,0.62,0.85,0.67,0.93,0.59,0.69,0.74,0.6,0.89,0.81,0.79,0.89,0.72,0.84,0.88,0.88,1.0,0.73,0.94,0.62,0.86,0.83,0.76,0.85,0.79,0.86,0.91,0.8,1.0,…,1.0,0.77,0.78,0.84,0.75,0.92,0.18,0.14,0.4,0.4,0.0,0.46,0.17,0.0,0.

In [19]:
X_train = low_SSW_high_SSB.select(cs.starts_with("0000")).transpose(column_names = low_SSW_high_SSB["locus"])
y_train = sample_train["group"].to_numpy()
y_label = sample_train["id"].to_numpy()
y_btk = sample_train["btk_id"].to_numpy()
X_train.write_csv(step_folder / "X_train.csv")
np.save(step_folder / "y_train.npy", y_train)
np.save(step_folder / "y_label.npy", y_label)
np.save(step_folder / "y_btk.npy", y_btk)